In [ ]:
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split

# Load Dataset

DATA_DIR = "donateacry_corpus"

records = []

# Loop through each category folder
for category in os.listdir(DATA_DIR):

    # Create the path of the category folder
    cat_folder = os.path.join(DATA_DIR, category)
    
    if os.path.isdir(cat_folder): # Check if the path is a folder
        wav_files = glob.glob(os.path.join(cat_folder, "*.wav")) # Get all WAV audio files inside the folder
        
        for file_path in wav_files: 
            records.append({"file_path": file_path, "label": category}) # Store the file path and its label

df = pd.DataFrame(records) # Convert the records list into a DataFrame

print(df["label"].value_counts()) # Display the number of samples in each class

# Split dataset into training and testing
train_df, test_df = train_test_split( df, test_size=0.20, random_state=42, stratify=df["label"] )

print("\n(Train Set: 80%) ")
print(train_df["label"].value_counts())

print("\n(Test Set: 20%)")
print(test_df["label"].value_counts())

label
hungry        382
discomfort     27
tired          24
belly_pain     16
burping         8
Name: count, dtype: int64

(Train Set: 80%) 
label
hungry        305
discomfort     22
tired          19
belly_pain     13
burping         6
Name: count, dtype: int64

(Test Set: 20%)
label
hungry        77
tired          5
discomfort     5
belly_pain     3
burping        2
Name: count, dtype: int64


## Features Extraction ##

In [ ]:
import numpy as np 
import librosa 
 
def extract_features(y, sr): 
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) 
    mfcc_mean = np.mean(mfcc, axis=1) 
    mfcc_std = np.std(mfcc, axis=1) 
     
    rms_mean = np.mean(librosa.feature.rms(y=y)) 
     
    zcr_mean = np.mean(librosa.feature.zero_crossing_rate(y=y)) 

    duration = np.array([len(y) / sr])

    f0 = libsora.yin(y, fmin=400, fmax=500, sr=sr)
    f0 = f0[np.isfinite(f0)]  # Remove invalid F0 values
    f0_mean = np.mean(f0)
    f0_std = np.std(f0)
    f0_min = np.min(f0)
    f0_max = np.max(f0)
    
    return np.hstack([mfcc_mean, mfcc_std, rms_mean, duration, zcr_mean, f0_mean, f0_std, f0_min, f0_max])

## Audio Data Augmentation ##

In [ ]:
import numpy as np
import librosa
import pandas as pd

# make audo augmentationnn
def augment_audio(y, sr):
    augmented_samples = []
    
    # Pitch shifting 
    augmented_samples.append(librosa.effects.pitch_shift(y=y, sr=sr, n_steps=1.5))
    augmented_samples.append(librosa.effects.pitch_shift(y=y, sr=sr, n_steps=-1.5))
    
    # Time stretching 
    augmented_samples.append(librosa.effects.time_stretch(y=y, rate=0.9))
    augmented_samples.append(librosa.effects.time_stretch(y=y, rate=1.1))
    
    # added  Gaussian white noise
    noise = np.random.randn(len(y))
    noise_factor = 0.005
    augmented_samples.append(y + (noise_factor * noise))
    
    return augmented_samples



In [5]:
# 2. Extract features for training set with augmentation on minority classes
X_train = []
y_train = []

for _, row in train_df.iterrows():
    y, sr = librosa.load(row['file_path'], sr=16000)
    
    # Extract features from original audio
    X_train.append(extract_features(y, sr))

    y_train.append(row['label'])

    
    # Augment minority classes only 
    if row['label'] != 'hungry':

        for aug_y in augment_audio(y, sr):

            X_train.append(extract_features(aug_y, sr))

            y_train.append(row['label'])

X_train = np.array(X_train)

y_train = np.array(y_train)



# 3. Extract features for test set without augmentation
X_test = []

y_test = []

print("Processing test set ")

for _, row in test_df.iterrows():
    y, sr = librosa.load(row['file_path'], sr=16000)
    X_test.append(extract_features(y, sr))
    y_test.append(row['label'])

X_test = np.array(X_test)
y_test = np.array(y_test)


# 4. Print shapes and class balances
print("\nSummary ")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

print("\nTraining set distribution after augmentation:")
print(pd.Series(y_train).value_counts())

print("\nTest set distribution:")
print(pd.Series(y_test).value_counts())

Processing test set 

Summary 
X_train shape: (665, 28)
y_train shape: (665,)
X_test shape:  (92, 28)
y_test shape:  (92,)

Training set distribution after augmentation:
hungry        305
discomfort    132
tired         114
belly_pain     78
burping        36
Name: count, dtype: int64

Test set distribution:
hungry        77
tired          5
discomfort     5
belly_pain     3
burping        2
Name: count, dtype: int64
